In [2]:
# 依据测试集的标签筛选训练集
import pandas as pd
import json
import random
random.seed(2023)
with open("/root/data1/liang/self-correct-retriever/data/train_data/cvg_split/test.json","r") as f:
    data_all = []
    lines = f.readlines()
    random.shuffle(lines)
    for line in lines:
        line = json.loads(line)
        data_all.append(line)
original_dataset = pd.DataFrame(data_all)
accusation_counts = original_dataset["char"].apply(lambda x: x).value_counts()
print(accusation_counts)

keys_with_values_less_than_5 = [key for key, value in accusation_counts.items() if value > 5]

print(len(keys_with_values_less_than_5))

filtered_data = original_dataset[original_dataset['char'].isin(keys_with_values_less_than_5)]
print(len(filtered_data))
data_all=[]
for i in range(len(filtered_data)):
    data_all.append(filtered_data.iloc[i].to_dict())
with open("/root/data1/liang/self-correct-retriever/data/knowledge_base/test_cvg.json", "w") as f:
    random.shuffle(data_all)
    for dic in data_all:
        json.dump(dic,f,ensure_ascii=False)
        f.write("\n")


char
危险驾驶罪         2744
盗窃罪           1946
故意伤害罪          822
诈骗罪            607
交通肇事罪          526
              ... 
运送他人偷越国境罪       11
销售伪劣产品罪         11
非国家工作人员受贿罪      11
集资诈骗罪           11
非法拘禁罪            9
Name: count, Length: 62, dtype: int64
62
9039


In [3]:
# 构建标签平衡的新数据集
import pandas as pd
import json
import random
# 假设您有一个原始数据集 original_dataset
# 示例数据格式如下：
with open("/root/data1/liang/self-correct-retriever/data/train_data/cvg_split/train.json","r") as f:
    data_all = []
    lines = f.readlines()
    for line in lines:
        line = json.loads(line)
        data_all.append(line)
original_dataset = pd.DataFrame(data_all)
original_dataset = original_dataset[original_dataset['char'].isin(keys_with_values_less_than_5)]
# 检查每个罪名的数量
accusation_counts = original_dataset["char"].apply(lambda x: x).value_counts()

# 设置每个罪名的目标数量（最多1000条）
target_count = 100

# 创建一个空的DataFrame来存储平衡后的数据
balanced_dataset = pd.DataFrame(columns=original_dataset.columns)

# 对每个罪名进行处理，确保数量不超过目标数量
for accusation, count in accusation_counts.items():
    subset = original_dataset[original_dataset["char"].apply(lambda x: x) == accusation]
    if count > target_count:
        subset = subset.sample(target_count)  # 如果数量超过目标数量，随机采样一部分数据
    balanced_dataset = pd.concat([balanced_dataset, subset])
print(len(balanced_dataset))
print(balanced_dataset["char"].apply(lambda x: x).value_counts())
data_all=[]
for i in range(len(balanced_dataset)):
    data_all.append(balanced_dataset.iloc[i].to_dict())
# 现在 balanced_dataset 就是平衡后的数据集，每个罪名不超过1000条
write_path = "/root/data1/liang/self-correct-retriever/data/knowledge_base/balanced_train_cvg.json"
with open(write_path, "w") as f:
    random.shuffle(data_all)
    for dic in data_all:
        json.dump(dic,f,ensure_ascii=False)
        f.write("\n")
print(len(data_all))   

6124
char
危险驾驶罪       100
偷越国境罪       100
失火罪         100
盗窃罪         100
抢夺罪         100
           ... 
组织卖淫罪        90
妨害信用卡管理罪     89
销售伪劣产品罪      86
集资诈骗罪        85
非法行医罪        81
Name: count, Length: 62, dtype: int64
6124
